# Principio de Responsabilidad Única (SRP) — Sistema de citas médicas

## Ejemplo que viola el SRP

La clase `Cita` de abajo mezcla **tres responsabilidades**: representar los datos de la cita, enviar el recordatorio al paciente y guardar la cita en la base de datos. Además calcula su propio costo.

In [3]:
class Cita:
    def __init__(self, paciente: str, medico: str, fecha: str, motivo: str) -> None:
        self.paciente = paciente
        self.medico = medico
        self.fecha = fecha
        self.motivo = motivo

    def enviar_recordatorio(self) -> str:
        mensaje = f"Recordatorio: {self.paciente} tiene cita con {self.medico} el {self.fecha}."
        print(mensaje)
        return mensaje

    def guardar_en_base_datos(self) -> None:
        print(f"[DB] Cita de {self.paciente} guardada.")

    def calcular_costo(self) -> float:
        tarifas = {"general": 50000, "especialista": 120000}
        return tarifas.get(self.motivo, 80000)


In [6]:
cita = Cita(paciente="Laura Gómez", medico="Dr. Pérez", fecha="2026-09-02", motivo="general")
cita.enviar_recordatorio()
cita.guardar_en_base_datos()
print("Costo:", cita.calcular_costo())

Recordatorio: Laura Gómez tiene cita con Dr. Pérez el 2026-09-02.
[DB] Cita de Laura Gómez guardada.
Costo: 50000


### Por qué esto es un problema

`Cita` tiene **tres razones distintas para cambiar**:

1. Si cambia el canal de recordatorio (de imprimir a enviar un SMS real), hay que tocar `Cita`.
2. Si cambia el motor de persistencia (de un `print` a una base de datos real), hay que tocar `Cita`.
3. Si cambian las tarifas o la política de precios, hay que tocar `Cita`.

## Versión corregida: una responsabilidad por clase

Se separa `Cita` (solo representa y modifica los datos propios de la cita) de tres colaboradores especializados:

- **`Cita`**: representa la cita y sus operaciones propias (reprogramar, resumir). Cambia solo si cambia el modelo de una cita.
- **`RecordatorioService`**: se encarga únicamente de notificar al paciente. Cambia solo si cambia el canal o formato de notificación.
- **`CitaRepository`**: se encarga únicamente de la persistencia. Cambia solo si cambia el motor de almacenamiento.
- **`FacturacionService`**: se encarga únicamente de calcular costos. Cambia solo si cambia la política de tarifas.

In [ ]:
class Cita:
    def __init__(self, paciente: str, medico: str, fecha: str, motivo: str) -> None:
        self.paciente = paciente
        self.medico = medico
        self.fecha = fecha
        self.motivo = motivo

    def reprogramar(self, nueva_fecha: str) -> None:
        self.fecha = nueva_fecha

    def resumen(self) -> str:
        return f"{self.paciente} con {self.medico} el {self.fecha} ({self.motivo})"


class RecordatorioService:
    def __init__(self, canal: str = "sms") -> None:
        self.canal = canal

    def enviar_recordatorio(self, cita: Cita) -> str:
        mensaje = f"[{self.canal.upper()}] Recordatorio para {cita.paciente}: cita el {cita.fecha} con {cita.medico}."
        print(mensaje)
        return mensaje


class CitaRepository:
    def __init__(self) -> None:
        self._citas_guardadas = []

    def guardar(self, cita: Cita) -> None:
        self._citas_guardadas.append(cita)
        print(f"[DB] Cita de {cita.paciente} guardada.")

    def buscar_por_paciente(self, paciente: str) -> list[Cita]:
        return [c for c in self._citas_guardadas if c.paciente == paciente]


class FacturacionService:
    TARIFAS = {"general": 50000, "especialista": 120000}

    def __init__(self, tarifa_default: float = 80000) -> None:
        self.tarifa_default = tarifa_default

    def calcular_costo(self, cita: Cita) -> float:
        return self.TARIFAS.get(cita.motivo, self.tarifa_default)


In [ ]:
cita = Cita(paciente="Laura Gómez", medico="Dr. Pérez", fecha="2026-09-02", motivo="general")

recordatorios = RecordatorioService(canal="sms")
repositorio = CitaRepository()
facturacion = FacturacionService()

recordatorios.enviar_recordatorio(cita)
repositorio.guardar(cita)
costo = facturacion.calcular_costo(cita)

print("Costo:", costo)
print("Citas encontradas para Laura Gómez:", [c.resumen() for c in repositorio.buscar_por_paciente("Laura Gómez")])

assert costo == 50000
assert len(repositorio.buscar_por_paciente("Laura Gómez")) == 1
print("¡Todo correcto!")


[SMS] Recordatorio para Laura Gómez: cita el 2026-09-02 con Dr. Pérez.
[DB] Cita de Laura Gómez guardada.
Costo: 50000
Recordatorios enviados: 1
Citas encontradas para Laura Gómez: ['Laura Gómez con Dr. Pérez el 2026-09-02 (general)']
¡Todo correcto!


### Análisis

Cada clase tiene ahora una única razón para cambiar:

- **`Cita`**: cambia solo si cambia qué información describe una cita.
- **`RecordatorioService`**: cambia solo si cambia el canal o formato del recordatorio (por ejemplo, pasar de SMS a WhatsApp).
- **`CitaRepository`**: cambia solo si cambia cómo se persisten las citas (por ejemplo, pasar de una lista en memoria a una base de datos real).
- **`FacturacionService`**: cambia solo si cambia la política de tarifas.

Modificar el canal de notificación ya no obliga a tocar `Cita`, ni cambiar las tarifas obliga a tocar `RecordatorioService`. Las responsabilidades quedaron desacopladas.